# Stroke Prediction Model

The dataset we are gonna be using this time is highly imbalanced(about 5% stroke data). Thus, we are gonna use SMOTE.

The dataset can can be downloaded from Kaggle using : https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset

---

# Data Analysis

In [117]:
import pandas as pd 
import numpy as np

### Importing the dataset

In [118]:
df = pd.read_csv("data/healthcare-dataset-stroke-data.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


**"id"**

The id feature is of no use for us. Let's remove it.

In [119]:
df = df.drop(columns="id")

**"gender"**

In [120]:
print(df["gender"].unique())
df[df["gender"] == "Other"]

<StringArray>
['Male', 'Female', 'Other']
Length: 3, dtype: str


,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
3116,Other,26.0,0,0,No,Private,Rural,143.33,22.4,formerly smoked,0


We only have one "Other" gender, let's remove it. And we can map the other two genders.

In [121]:
df = df.drop(3116)
print(df.shape)
df["gender"] = df["gender"].map({"Male" : 1, "Female" : 0})

(5109, 11)


**"hypertension & heart_disease"**

In [122]:
print(df["hypertension"].unique())
print(df["heart_disease"].unique())

[0 1]
[1 0]


**"ever_married"**

In [123]:
print(df["ever_married"].unique())
df["ever_married"] = df["ever_married"].map({"Yes" : 1, "No" : 0})

<StringArray>
['Yes', 'No']
Length: 2, dtype: str


**"work_type"**

In [124]:
print(df["work_type"].unique())
df["work_type"].value_counts()

<StringArray>
['Private', 'Self-employed', 'Govt_job', 'children', 'Never_worked']
Length: 5, dtype: str


work_type
Private          2924
Self-employed     819
children          687
Govt_job          657
Never_worked       22
Name: count, dtype: int64

There are some children too. So we can combine the children with the Never_worked.

In [125]:
df["work_type"] = df["work_type"].replace({"children" : "Never_worked"})
df["work_type"].value_counts()

work_type
Private          2924
Self-employed     819
Never_worked      709
Govt_job          657
Name: count, dtype: int64

On the rest, we can use OHE.

**"Residence_type"**

In [126]:
print(df["Residence_type"].unique())
# Let's map Urban as 1 and Rural as 0.
df["Residence_type"] = df["Residence_type"].map({"Urban" : 1, "Rural" : 0})

<StringArray>
['Urban', 'Rural']
Length: 2, dtype: str


***"smoking_status"***

In [127]:
df["smoking_status"].value_counts()

smoking_status
never smoked       1892
Unknown            1544
formerly smoked     884
smokes              789
Name: count, dtype: int64

We can use OHE on this too.

---

### Overview

In [128]:
df.info()

<class 'pandas.DataFrame'>
Index: 5109 entries, 0 to 5109
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             5109 non-null   int64  
 1   age                5109 non-null   float64
 2   hypertension       5109 non-null   int64  
 3   heart_disease      5109 non-null   int64  
 4   ever_married       5109 non-null   int64  
 5   work_type          5109 non-null   str    
 6   Residence_type     5109 non-null   int64  
 7   avg_glucose_level  5109 non-null   float64
 8   bmi                4908 non-null   float64
 9   smoking_status     5109 non-null   str    
 10  stroke             5109 non-null   int64  
dtypes: float64(3), int64(6), str(2)
memory usage: 479.0 KB


The bmi column has some null values.

In [129]:
df["bmi"][800:808]

800    34.9
801    35.0
802    28.5
803    32.3
804    23.9
805    52.3
806    26.4
807    20.9
Name: bmi, dtype: float64

Let's see if we can replace the null values with the mean of this column.

In [130]:
df["bmi"].mean()

np.float64(28.894559902200488)

The mean seems to be....reasonable, let's replace the null values with it.

In [131]:
df["bmi"] = df["bmi"].fillna(df["bmi"].mean())

In [132]:
df.isna().sum()

gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64

In [133]:
df.describe()

,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,stroke
count,5109.000000,5109.000000,5109.000000,5109.000000,5109.000000,5109.000000,5109.000000,5109.000000,5109.000000
mean,0.413975,43.229986,0.097475,0.054022,0.656293,0.508123,106.140399,28.894560,0.048738
std,0.492592,22.613575,0.296633,0.226084,0.474991,0.499983,45.285004,7.698235,0.215340
min,0.000000,0.080000,0.000000,0.000000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,77.240000,23.800000,0.000000
50%,0.000000,45.000000,0.000000,0.000000,1.000000,1.000000,91.880000,28.400000,0.000000
75%,1.000000,61.000000,0.000000,0.000000,1.000000,1.000000,114.090000,32.800000,0.000000
max,1.000000,82.000000,1.000000,1.000000,1.000000,1.000000,271.740000,97.600000,1.000000


One more thing that we would need to do is to Scale the age, avg_glucose_level, and bmi column.

---

### Pending Tasks

- Scale the age, avg_glucose_level, and bmi column.
- Apply OHE on work_type, smoking_status.

---

## Saving the Dataframe

In [134]:
df.to_csv("data/cleaned-stroke-dataset.csv", index = False)